In [ ]:
#@title Cell 50.1 — Purpose, study design and limitations
from IPython.display import display, Markdown
display(Markdown(r"""
# NB50 — Genomic-group-specific prediction uncertainty in BMC-N, BMC-T and BMC-AT

**Question:** Within an antibiotic, do pathogens assigned to different genomic-similarity groups show different **prediction-error magnitudes**, **95% predictive coverage**, and **upper-tail misses**?

**Inputs:** The *saved held-out* BMC predictions from NB41, NB42, NB43; the *saved genomic group assignments* from NB47. **No new prediction model, calibration fit, or genomic clustering.**

**Design:** For every antibiotic, original held-out fold, and fold-specific genomic group, evaluate all three BMC predictive intervals against the same observed MICs. We only use the original 50,460 distinct held-out pathogen–antibiotic MIC observations, not the overlapping inner calibration records.

**Important limitations:**
- NB47's four genomic clusters were defined using genomic embeddings without MIC outcomes, with training-only k-means per outer fold.
- **Genomic-group labels are not aligned across folds.** Group 0 in fold 1 is not assumed equivalent to group 0 in fold 2. We evaluate group differences *within each fold*; we never pool group IDs across folds.
- Group sizes vary, so conditional coverage can fluctuate from sampling variation alone. All cells are reported, but the primary descriptive comparison requires **≥30 held-out MIC records** per group. Wilson intervals show the uncertainty of observed proportions.
- Observed differences do **not** prove genetic causation or that a new model would improve individual-pathogen predictions. No hypothesis-driven model selection is made from these held-out results.
- All MICs are quantitative **log₂(mg/L)**. No common resistance breakpoint is assumed for the 26 antibiotics.

**Outputs:** All fold–drug–group metrics; within-fold group-spread summaries; four-focus-antibiotic plots; and a validation audit in `03_Results/50_BMC_Genomic_Group_Uncertainty_Audit`.
"""))


In [ ]:
#@title Cell 50.2 — Automatically locate four existing input files
from pathlib import Path
import os, hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from IPython.display import display

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
except ImportError:
    pass
ROOT=Path(os.environ.get('BMC_PROJECT_ROOT','/content/drive/MyDrive/RefMIC'))
if not ROOT.is_dir():
    raise FileNotFoundError(f'RefMIC folder not found: {ROOT}. Set BMC_PROJECT_ROOT to your project directory.')
SEARCH=Path('/content/drive') if Path('/content/drive').is_dir() else ROOT
OUT=ROOT/'03_Results'/'50_BMC_Genomic_Group_Uncertainty_Audit'
OUT.mkdir(parents=True,exist_ok=True)
FILES={
 'N':('41_BMC_N_Pathogen_Out_Validation','41_BMC_N_independent_outer_test_predictions.csv.gz'),
 'T':('42_BMC_T_Pathogen_Out_Validation','42_BMC_T_independent_outer_test_predictions.csv.gz'),
 'AT':('43_BMC_AT_Pathogen_Out_Validation','43_BMC_AT_independent_outer_test_predictions.csv.gz'),
 'GROUP':('47_ModelC_Genomic_Group_Slope_Audit','47_fold_genomic_group_assignments.csv.gz'),
}
def digest(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for part in iter(lambda:f.read(4*1024*1024),b''):
            h.update(part)
    return h.hexdigest()
def locate(folder,name):
    preferred=ROOT/'03_Results'/folder/name
    if preferred.is_file():
        return preferred
    hits=[]
    for path,dirs,files in os.walk(SEARCH):
        dirs[:]=[d for d in dirs if d not in {'.git','.Trashes','.Trash-1000','node_modules','__pycache__'}]
        if name in files: hits.append(Path(path)/name)
    if not hits:
        raise FileNotFoundError(f'{name} not found under {SEARCH}. Please finish NB41–43 and NB47 before NB50.')
    if len(hits)>1 and len({digest(p) for p in hits})>1:
        raise ValueError(f'Different copies of {name} found. Place the authoritative one in {preferred.parent}.\n'+'\n'.join(map(str,hits)))
    return sorted(hits,key=lambda p:(0 if ROOT in p.parents else 1,len(p.parts),str(p)))[0]
PATH={k:locate(folder,name) for k,(folder,name) in FILES.items()}
for label,p in PATH.items(): print(label+':',p)
print('NB50 results:',OUT)


In [ ]:
#@title Cell 50.3 — Join and validate the original held-out predictions and genomic groups
KEYS=['interaction_row','biosample','antibiotic','outer_fold']
IDENTITY=['interaction_row','biosample','antibiotic','outer_fold','observed_log2_mic','model_c_outer_test_predicted_log2_mic']
MODELS={'N':'bmc_n','T':'bmc_t','AT':'bmc_at'}
all_pred={}
for model,prefix in MODELS.items():
    tab=pd.read_csv(PATH[model],compression='gzip')
    required=set(IDENTITY)|{f'{prefix}_{end}95_log2' for end in ['lower','upper']}|{f'{prefix}_median_log2'}
    missing=required-set(tab)
    if missing:
        raise ValueError(f'BMC-{model} missing required columns: {sorted(missing)}')
    if (len(tab),tab.biosample.nunique(),tab.antibiotic.nunique(),tab.interaction_row.nunique())!=(50460,9058,26,50460):
        raise ValueError(f'BMC-{model} has unexpected held-out dimensions')
    if tab.duplicated(KEYS).any() or tab.duplicated(['biosample','antibiotic']).any():
        raise ValueError(f'BMC-{model}: duplicate held-out predictions')
    if tab.groupby('biosample').outer_fold.nunique().max()!=1 or set(tab.outer_fold)!=set(range(1,6)):
        raise ValueError(f'BMC-{model}: invalid held-out pathogen folds')
    all_pred[model]=tab[IDENTITY+[f'{prefix}_{k}_log2' for k in ['lower95','median','upper95']]].copy()

pred=all_pred['N'].copy()
for model in ['T','AT']:
    candidate=all_pred[model].copy()
    # Audit using original row identities and MIC outcomes before merging intervals.
    audit=pred[IDENTITY].merge(candidate[IDENTITY],on=KEYS,how='outer',indicator=True,suffixes=('_ref','_new'),validate='one_to_one')
    if len(audit)!=50460 or not audit._merge.eq('both').all():
        raise ValueError(f'BMC-{model}: unmatched test observations')
    for var in ['observed_log2_mic','model_c_outer_test_predicted_log2_mic']:
        if not np.allclose(audit[f'{var}_ref'],audit[f'{var}_new'],atol=1e-9,rtol=0):
            raise ValueError(f'BMC-{model}: changed {var} values')
    pred=pred.merge(candidate[KEYS+[f'{MODELS[model]}_{k}_log2' for k in ['lower95','median','upper95']]],
                    on=KEYS,how='left',validate='one_to_one')
assign=pd.read_csv(PATH['GROUP'],compression='gzip')
needed={'outer_fold','biosample','partition','genomic_group'}
if needed-set(assign):
    raise ValueError(f'Genomic group assignment missing columns: {sorted(needed-set(assign))}')
if len(assign)!=5*9058 or assign.duplicated(['outer_fold','biosample']).any():
    raise ValueError('NB47 genomic assignment table must contain one assignment for each BioSample in each fold')
if set(assign.outer_fold)!=set(range(1,6)) or set(assign.genomic_group.astype(int))!=set(range(4)):
    raise ValueError('Unexpected folds or number of genomic groups')
heldout=assign.loc[assign.partition.eq('test'),['outer_fold','biosample','genomic_group']].copy()
if len(heldout)!=9058 or heldout.biosample.nunique()!=9058:
    raise ValueError('NB47 must contain exactly 9,058 unique test BioSamples across five folds')
if not set(heldout.biosample)==set(pred.biosample):
    raise ValueError('Genomic groups do not match the held-out BioSample population')
pred=pred.merge(heldout,on=['outer_fold','biosample'],how='left',validate='many_to_one')
if pred.genomic_group.isna().any() or len(pred)!=50460:
    raise ValueError('Some original held-out observations do not have a genomic assignment')
pred['genomic_group']=pred.genomic_group.astype(int)
cols=['observed_log2_mic','model_c_outer_test_predicted_log2_mic']+[f'{prefix}_{k}_log2' for prefix in MODELS.values() for k in ['lower95','median','upper95']]
if not np.isfinite(pred[cols].to_numpy(dtype=float)).all():
    raise ValueError('Nonfinite observed MIC, prediction, or intervals')
for prefix in MODELS.values():
    if not (pred[f'{prefix}_lower95_log2']<pred[f'{prefix}_upper95_log2']).all():
        raise ValueError(f'{prefix}: invalid interval bounds')
    if not pred[f'{prefix}_median_log2'].between(pred[f'{prefix}_lower95_log2'],pred[f'{prefix}_upper95_log2']).all():
        raise ValueError(f'{prefix}: median lies outside its interval')
print('Verified 50,460 independent MIC observations, 9,058 BioSamples, 26 antibiotics.')
print('All three models evaluated the identical held-out observations and five folds.')
print('Genomic groups are the NB47 training-defined, MIC-independent assignments (test partition only).')
display(pred.groupby(['outer_fold','genomic_group'],as_index=False).agg(n_MIC=('interaction_row','size'),n_pathogens=('biosample','nunique')))


In [ ]:
#@title Cell 50.4 — Measure prediction error and conditional coverage within each group
MIN_GROUP_TEST=30  # Descriptive display threshold only, never fitted or tuned on test outcomes.

def wilson(k,n,z=float(norm.ppf(.975))):
    p=k/n
    d=1+z*z/n
    c=(p+z*z/(2*n))/d
    h=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return (c-h,c+h)

long_rows=[]
for model,prefix in MODELS.items():
    t=pred[KEYS+['genomic_group','observed_log2_mic','model_c_outer_test_predicted_log2_mic']].copy()
    t['model']='BMC-'+model
    t['lower95_log2']=pred[f'{prefix}_lower95_log2'].to_numpy()
    t['median_log2']=pred[f'{prefix}_median_log2'].to_numpy()
    t['upper95_log2']=pred[f'{prefix}_upper95_log2'].to_numpy()
    t['model_c_error_log2']=t.observed_log2_mic-t.model_c_outer_test_predicted_log2_mic
    t['above_upper95']=(t.observed_log2_mic>t.upper95_log2)
    t['below_lower95']=(t.observed_log2_mic<t.lower95_log2)
    t['covered_95pct']=(~t.above_upper95)&(~t.below_lower95)
    t['interval_width_log2']=t.upper95_log2-t.lower95_log2
    long_rows.append(t)
long=pd.concat(long_rows,ignore_index=True)
if len(long)!=3*50460:
    raise ValueError('Unexpected number of combined held-out observations')

def summarize_group(d):
    n=len(d)
    kcov=int(d.covered_95pct.sum()); kupper=int(d.above_upper95.sum()); klower=int(d.below_lower95.sum())
    cc=wilson(kcov,n);uc=wilson(kupper,n)
    err=d.model_c_error_log2.to_numpy(dtype=float)
    return pd.Series({
        'n_test':n,'n_pathogens':d.biosample.nunique(),
        'model_c_residual_MAE_log2':np.mean(np.abs(err)),
        'model_c_residual_RMSE_log2':np.sqrt(np.mean(err**2)),
        'observed_MiC_sd_log2':d.observed_log2_mic.std(ddof=0),
        'n_error_ge_2_log2':int(np.sum(err>=2)),
        'n_error_ge_4_log2':int(np.sum(err>=4)),
        'coverage_95pct':kcov/n,'coverage_wilson_low':cc[0],'coverage_wilson_high':cc[1],
        'n_above_upper95':kupper,'above_upper95_pct':100*kupper/n,
        'upper_miss_wilson_low_pct':100*uc[0],'upper_miss_wilson_high_pct':100*uc[1],
        'n_below_lower95':klower,'below_lower95_pct':100*klower/n,
        'mean_width_log2':d.interval_width_log2.mean(),
    })

GROUP_KEYS=['model','antibiotic','outer_fold','genomic_group']
groups=long.groupby(GROUP_KEYS,sort=True).apply(summarize_group).reset_index()
groups['well_supported']=groups.n_test>=MIN_GROUP_TEST
if not (groups.groupby('model').n_test.sum()==50460).all():
    raise ValueError('Group counts do not reproduce 50,460 observations per BMC model')
# Confirm Model C MAE is independent of which Bayesian likelihood is being evaluated.
mae_pivot=groups.pivot(index=['antibiotic','outer_fold','genomic_group'],columns='model',values='model_c_residual_MAE_log2')
if (mae_pivot.max(axis=1)-mae_pivot.min(axis=1)).max()>1e-9:
    raise ValueError('Original Model C residuals changed across model comparisons')
print('Fold-specific genomic group metrics:',len(groups),'records; supported groups:',int(groups.well_supported.sum()))
print('Each observed group has its own Wilson intervals; low-count groups remain in the export.')
FOCUS=['ceftazidime','cefepime','ciprofloxacin','meropenem']
display(groups[(groups.model.eq('BMC-AT'))&groups.antibiotic.isin(FOCUS)][
    ['antibiotic','outer_fold','genomic_group','n_test','model_c_residual_MAE_log2','coverage_95pct','n_above_upper95','above_upper95_pct','mean_width_log2','well_supported']
].round(3).head(32))


In [ ]:
#@title Cell 50.5 — Summarise between-group uncertainty differences within each antibiotic and fold
# Fold-specific comparisons avoid incorrect pooling of arbitrary genomic-group labels.
within_rows=[]
for (model,drug,fold),d0 in groups.groupby(['model','antibiotic','outer_fold'],sort=True):
    d=d0[d0.well_supported].copy()
    obj={'model':model,'antibiotic':drug,'outer_fold':int(fold),
         'n_groups_present':len(d0),'n_groups_supported':len(d),
         'n_MIC_supported':int(d.n_test.sum()),'n_MIC_all_groups':int(d0.n_test.sum())}
    for name,column in [('MAE','model_c_residual_MAE_log2'),('coverage','coverage_95pct'),
                        ('upper_miss_pct','above_upper95_pct'),('width','mean_width_log2')]:
        obj[f'{name}_spread']=float(d[column].max()-d[column].min()) if len(d)>=2 else np.nan
    within_rows.append(obj)
within=pd.DataFrame(within_rows)
if len(within)!=3*26*5:
    raise ValueError('Expected 390 model–antibiotic–fold summaries')

def summarize_antibiotic(t):
    e=t[t.n_groups_supported.ge(2)]
    return pd.Series({
        'n_folds_with_two_supported_groups':len(e),
        'min_groups_supported':int(t.n_groups_supported.min()),
        'median_groups_supported':float(t.n_groups_supported.median()),
        'median_within_fold_MAE_spread_log2':e.MAE_spread.median() if len(e) else np.nan,
        'median_within_fold_coverage_spread_pct':100*e.coverage_spread.median() if len(e) else np.nan,
        'median_within_fold_upper_miss_spread_pct':e.upper_miss_pct_spread.median() if len(e) else np.nan,
        'median_within_fold_width_spread_log2':e.width_spread.median() if len(e) else np.nan,
    })
abx_summary=within.groupby(['model','antibiotic'],sort=True).apply(summarize_antibiotic).reset_index()
if len(abx_summary)!=3*26:raise ValueError('Missing antibiotic–model summary')
print('Focus antibiotic group differences — within-fold medians, not pooled group IDs:')
display(abx_summary[abx_summary.antibiotic.isin(FOCUS)].sort_values(['antibiotic','model']).round(3))
print('All 26 antibiotics — BMC-AT, ranked by within-fold group error magnitude difference:')
display(abx_summary[abx_summary.model.eq('BMC-AT')].sort_values('median_within_fold_MAE_spread_log2',ascending=False).round(3))
print('Caution: each displayed group spread is influenced by group sample size and is not a significance test.')


In [ ]:
#@title Cell 50.6 — Figure 1: compare genomic-group uncertainty differences for 26 antibiotics
# A small number of supported groups can create an unstable spread, so show the number of folds.
tab=abx_summary.loc[abx_summary.model.eq('BMC-AT')].sort_values('median_within_fold_MAE_spread_log2',ascending=True).reset_index(drop=True)
fig,ax=plt.subplots(figsize=(9,11))
valid=tab.n_folds_with_two_supported_groups.gt(0)
ax.barh(np.arange(len(tab))[valid],tab.loc[valid,'median_within_fold_MAE_spread_log2'],color='steelblue',alpha=.8)
ax.set_yticks(np.arange(len(tab)),tab.antibiotic,fontsize=9)
for i,r in tab.iterrows():
    ax.annotate(f"{int(r.n_folds_with_two_supported_groups)}/5 folds",(0 if not np.isfinite(r.median_within_fold_MAE_spread_log2) else r.median_within_fold_MAE_spread_log2,i),
                xytext=(5,0),textcoords='offset points',ha='left',va='center',fontsize=7)
ax.set_xlabel('Median within-fold difference in Model C residual MAE (log₂ MIC)')
ax.set_title('Genomic-group variation within antibiotic — exploratory')
ax.grid(axis='x',alpha=.2)
fig.tight_layout()
p=OUT/'50_Figure1_26_Antibiotic_Within_Fold_Genomic_Group_MAE_Spread.png'
fig.savefig(p,dpi=170,bbox_inches='tight')
plt.show()
print('Saved:',p)


In [ ]:
#@title Cell 50.7 — Figure 2: compare three BMC interval coverages within four focus antibiotics
for drug in FOCUS:
    d=groups[groups.antibiotic.eq(drug)].copy()
    # Equal sets of genomic group labels for all BMC models in this fold.
    keys=d[['outer_fold','genomic_group']].drop_duplicates().sort_values(['outer_fold','genomic_group']).reset_index(drop=True)
    labels=[f'F{int(f)} G{int(g)}' for f,g in zip(keys.outer_fold,keys.genomic_group)]
    fig,axes=plt.subplots(2,1,figsize=(13,8),sharex=True)
    for model,color,mark in [('BMC-N','tab:blue','o'),('BMC-T','tab:orange','s'),('BMC-AT','tab:green','^')]:
        q=keys.merge(d[d.model.eq(model)][['outer_fold','genomic_group','n_test','well_supported','coverage_95pct','above_upper95_pct']],
                     on=['outer_fold','genomic_group'],how='left',validate='one_to_one')
        supported=q.well_supported.fillna(False)
        axes[0].plot(np.where(supported)[0],100*q.loc[supported,'coverage_95pct'],label=model,color=color,marker=mark,lw=1,markersize=4)
        axes[1].plot(np.where(supported)[0],q.loc[supported,'above_upper95_pct'],label=model,color=color,marker=mark,lw=1,markersize=4)
    axes[0].axhline(95,ls='--',color='gray',lw=1,label='Nominal 95%')
    axes[0].set_ylabel('95% interval coverage (%)')
    axes[1].set_ylabel('Above upper limit (%)')
    axes[1].set_xlabel('Outer validation fold and local genomic-group label (not aligned across folds)')
    for ax in axes: ax.grid(axis='y',alpha=.2)
    axes[0].legend(ncol=4,fontsize=9,loc='best')
    axes[1].set_xticks(np.arange(len(keys)),labels,rotation=65,ha='right',fontsize=8)
    axes[0].set_title(f'{drug}: within-antibiotic genomic-group conditional uncertainty\nOnly groups with ≥{MIN_GROUP_TEST} held-out MICs shown')
    fig.tight_layout()
    p=OUT/f'50_Figure2_Conditional_Intervals_{drug}.png'
    fig.savefig(p,dpi=165,bbox_inches='tight')
    plt.show()
    print('Saved:',p)
print('Plot 2 excludes low-count cells from the figure only; they remain in the output CSV.')


In [ ]:
#@title Cell 50.8 — Save tables and audit; report what remains unresolved
files=[
    ('50_independent_test_predictions_with_genomic_groups.csv.gz',pred),
    ('50_fold_antibiotic_group_model_conditional_uncertainty.csv',groups),
    ('50_within_fold_genomic_group_spreads.csv',within),
    ('50_26_antibiotic_model_group_uncertainty_summary.csv',abx_summary),
]
for name,df in files:
    dest=OUT/name
    df.to_csv(dest,index=False,compression='gzip' if dest.suffix=='.gz' else None)
    print('Saved:',dest)
report={
 'n_distinct_held_out_MICS':int(pred.interaction_row.nunique()),
 'n_BioSamples':int(pred.biosample.nunique()),
 'n_antibiotics':int(pred.antibiotic.nunique()),
 'n_BMC_models':len(MODELS),
 'n_outer_folds':int(pred.outer_fold.nunique()),
 'n_genomic_groups_per_outer_fold':4,
 'min_group_test_n_for_descriptive_comparison':MIN_GROUP_TEST,
 'n_fold_drug_group_model_cells':len(groups),
 'n_fold_drug_group_model_cells_supported':int(groups.well_supported.sum()),
 'no_recalibration_or_new_clustering':True,
 'no_pooled_group_IDs_across_folds':True,
 'all_scores_held_out':True,
 'note':'Conditional group comparisons are descriptive and do not establish causation or external validity.',
}
(OUT/'50_validation_audit.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('NB50 complete. Share Cell 50.5, especially the BMC-AT 26-antibiotic table.')
print('We will decide the manuscript interpretation, then proceed to NB51: simple empirical interval benchmark and interval scores.')
